# H2O AutoML
#### It loads the dataset, converts it into an H2OFrame, preprocesses categorical features, and then runs H2O's AutoML to automatically train and evaluate multiple models. Finally, it prints the leaderboard of trained models and saves the best-performing model.

##### Install necessary libraries if not already installed Run these in the terminal or command prompt before executing the script


In [ ]:
!pip3 install h2o pandas

In [1]:
import h2o  # Import H2O for machine learning
import pandas as pd  # Import pandas for data manipulation
from h2o.automl import H2OAutoML  # Import H2O's AutoML for automated model training

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: java version "21.0.4" 2024-07-16 LTS; Java(TM) SE Runtime Environment (build 21.0.4+8-LTS-274); Java HotSpot(TM) 64-Bit Server VM (build 21.0.4+8-LTS-274, mixed mode, sharing)
  Starting server from /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/cc/wqgzr72j1c9c8wc6sysbcblc0000gn/T/tmpgovhs74q
  JVM stdout: /var/folders/cc/wqgzr72j1c9c8wc6sysbcblc0000gn/T/tmpgovhs74q/h2o_prasadkatkade_started_from_python.out
  JVM stderr: /var/folders/cc/wqgzr72j1c9c8wc6sysbcblc0000gn/T/tmpgovhs74q/h2o_prasadkatkade_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,America/Chicago
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.6
H2O_cluster_version_age:,6 months and 1 day
H2O_cluster_name:,H2O_from_python_prasadkatkade_dysbjm
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.983 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


##### Load dataset into a Pandas DataFrame, Convert the Pandas DataFrame into an H2OFrame, required for H2O models

In [2]:
df = pd.read_csv("sales_data.csv") 
df_h2o = h2o.H2OFrame(df)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


#####  Convert categorical features into "factors" to ensure they are handled correctly by H2O

In [3]:
for col in df_h2o.columns:
    if df_h2o[col].isfactor()[0]:  
        df_h2o[col] = df_h2o[col].asfactor()  

df_h2o["SIM_DATE_year"] = df_h2o["SIM_DATE"].year()
df_h2o["SIM_DATE_month"] = df_h2o["SIM_DATE"].month()
df_h2o["SIM_DATE_day"] = df_h2o["SIM_DATE"].day()

#####  Define the target variable (the value we want to predict), Define predictor variables (all other columns except the target)

In [4]:
target = "NET_PRICE"
features = [col for col in df_h2o.columns if col != target]

##### Initialize H2O AutoML with a limit of 10 models and a random seed for reproducibility


In [5]:
aml = H2OAutoML(max_models=10, seed=42)

##### Train the AutoML models using the training dataset


In [6]:
aml.train(x=features, y=target, training_frame=df_h2o)

AutoML progress: |
15:49:25.565: AutoML: XGBoost is not available; skipping it.
15:49:25.578: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]


15:49:26.169: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]

███
15:49:29.34: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]
15:49:29.560: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day, COUNTRY, STORAGE_LOCATION, DISTRIBUTION_CHANNEL, ID]

█
15:49:30.224: _train param, Dropping bad and constant columns: [CURRENCY, MATERIAL_TYPE, SIM_DATE_year, SIM_DATE_month, UNIT, SIM_DATE_day,

Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_5_AutoML_1_20250503_154925


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    88                 88                          52143                  5            6            5.95455       12            52            35.5455

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 8.580415550203959e-09
RMSE: 9.263053249444245e-05
MAE: 7.027122218002857e-05
RMSLE: 1.4276870941753943e-05
Mean Residual Deviance: 8.580415550203959e-09

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 7.765414293928194e-05
RMSE: 0.008812158812645283
MAE: 0.0008950703998776182
RMSLE: 0.0015427739002059208
Mean Residual Deviance: 7.765414293928194e-05

Cross-Validation Metrics Summary: 
                        mean         sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  -----------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan          0            nan           nan           nan           nan           nan
loglikelihood           nan          0            nan           nan           nan           nan           nan
mae                     0.000875604  0.00059761   0.00161508    0.00042969    0.000387052   0.00143418    0.000512017
mean_residual_deviance  7.76388e-05  7.94137e-05  0.000136139   8.61521e-06   1.89036e-05   0.000187073   3.74639e-05
mse                     7.76388e-05  7.94137e-05  0.000136139   8.61521e-06   1.89036e-05   0.000187073   3.74639e-05
r2                      0.999909     9.38628e-05  0.999842      0.99999       0.999977      0.999778      0.999957
residual_deviance       7.76388e-05  7.94137e-05  0.000136139   8.61521e-06   1.89036e-05   0.000187073   3.74639e-05
rmse                    0.00774981   0.00468764   0.0116678     0.00293517    0.00434783    0.0136775     0.00612077
rmsle                   0.00131561   0.000900322  0.00216789    0.000489879   0.000529451   0.00237643    0.00101442

Scoring History: 
    timestamp            duration    number_of_trees    training_rmse    training_mae    training_deviance
--  -------------------  ----------  -----------------  ---------------  --------------  -------------------
    2025-05-03 15:49:32  0.276 sec   0                  0.922807         0.728716        0.851572
    2025-05-03 15:49:32  0.293 sec   5                  0.544998         0.430257        0.297023
    2025-05-03 15:49:32  0.299 sec   10                 0.321955         0.254113        0.103655
    2025-05-03 15:49:32  0.305 sec   15                 0.190109         0.150046        0.0361414
    2025-05-03 15:49:32  0.312 sec   20                 0.112355         0.088644        0.0126236
    2025-05-03 15:49:32  0.317 sec   25                 0.0663429        0.0523412       0.00440138
    2025-05-03 15:49:32  0.323 sec   30                 0.0391802        0.0309107       0.00153509
    2025-05-03 15:49:32  0.329 sec   35                 0.0231515        0.0182594       0.000535993
    2025-05-03 15:49:32  0.336 sec   40                 0.0136823        0.0107859       0.000187206
    2025-05-03 15:49:32  0.343 sec   45                 0.00807873       0.00636889      6.52658e-05
    2025-05-03 15:49:32  0.350 sec   50                 0.00477153       0.00376089      2.27675e-05
    2025-05-03 15:49:32  0.357 sec   55                 0.00282012       0.00222158      7.95306e-06
    2025-05-03 15:49:32  0.364 sec   60                 0.00166805       0.00131305      2.78239e-06
    2025-05-03 15:49:32  0.372 sec   65                 0.000987202      0.000776485     9.74567e-07
   

##### Print the leaderboard displaying all trained models ranked by performance, and save the best model

In [7]:
print(aml.leaderboard)
model_path = h2o.save_model(aml.leader, path="./best_price_model", force=True)
print("Model saved at:", model_path)

model_id                                                       rmse          mse         mae       rmsle    mean_residual_deviance
GBM_5_AutoML_1_20250503_154925                           0.00881216  7.76541e-05  0.00089507  0.00154277               7.76541e-05
StackedEnsemble_BestOfFamily_1_AutoML_1_20250503_154925  0.0147886   0.000218704  0.00384733  0.00265128               0.000218704
StackedEnsemble_AllModels_1_AutoML_1_20250503_154925     0.0181092   0.000327943  0.00397208  0.00303333               0.000327943
GBM_grid_1_AutoML_1_20250503_154925_model_1              0.0233421   0.000544853  0.00257628  0.00382644               0.000544853
GBM_4_AutoML_1_20250503_154925                           0.024173    0.000584336  0.00476904  0.00359637               0.000584336
GBM_2_AutoML_1_20250503_154925                           0.0344217   0.00118485   0.00629241  0.00548451               0.00118485
GBM_3_AutoML_1_20250503_154925                           0.0352667   0.00124374   0.